In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(
    "../data/processed/CCPP_clean.csv"
)

df.head()

,AT,V,AP,RH,PE
0,14.96,41.76,1024.07,73.17,463.26
1,25.18,62.96,1020.04,59.08,444.37
2,5.11,39.40,1012.16,92.14,488.56
3,20.86,57.32,1010.24,76.64,446.48
4,10.82,37.50,1009.23,96.62,473.90


In [3]:
features = ["AT", "V", "AP", "RH"]
target = "PE"

X = df[features].copy()
y = df[target].copy()

In [4]:
X["AT_V_interaction"] = X["AT"] * X["V"]

In [5]:
X["AP_RH_ratio"] = X["AP"] / X["RH"]

In [6]:
X["AT_squared"] = X["AT"] ** 2

In [7]:
X.head()

,AT,V,AP,RH,AT_V_interaction,AP_RH_ratio,AT_squared
0,14.96,41.76,1024.07,73.17,624.7296,13.995763,223.8016
1,25.18,62.96,1020.04,59.08,1585.3328,17.265403,634.0324
2,5.11,39.40,1012.16,92.14,201.3340,10.985023,26.1121
3,20.86,57.32,1010.24,76.64,1195.6952,13.181628,435.1396
4,10.82,37.50,1009.23,96.62,405.7500,10.445353,117.0724


In [8]:
X.corr()

,AT,V,AP,RH,AT_V_interaction,AP_RH_ratio,AT_squared
AT,1.000000,0.843689,-0.508222,-0.543947,0.963583,0.505546,0.982552
V,0.843689,1.000000,-0.415718,-0.312214,0.943651,0.282141,0.842156
AP,-0.508222,-0.415718,1.000000,0.101631,-0.488074,-0.086993,-0.504911
RH,-0.543947,-0.312214,0.101631,1.000000,-0.475794,-0.957365,-0.568407
AT_V_interaction,0.963583,0.943651,-0.488074,-0.475794,1.000000,0.442566,0.970139
AP_RH_ratio,0.505546,0.282141,-0.086993,-0.957365,0.442566,1.000000,0.538909
AT_squared,0.982552,0.842156,-0.504911,-0.568407,0.970139,0.538909,1.000000


In [9]:
feature_target_corr = (
    pd.concat([X, y], axis=1)
    .corr()["PE"]
    .drop("PE")
    .sort_values(key=abs, ascending=False)
)

feature_target_corr

AT                 -0.947908
AT_V_interaction   -0.934471
AT_squared         -0.914546
V                  -0.869900
AP                  0.518687
RH                  0.391175
AP_RH_ratio        -0.356661
Name: PE, dtype: float64

## Analisis Kandidat Feature Engineering

Tiga fitur turunan diuji sebagai kandidat:

- `AT_squared = AT²`
- `AT_V_interaction = AT × V`
- `AP_RH_ratio = AP / RH`

Hasil korelasi terhadap target `PE` menunjukkan:

- `AT` memiliki korelasi paling kuat, yaitu sekitar -0.948.
- `AT_V_interaction` memiliki korelasi sekitar -0.934.
- `AT_squared` memiliki korelasi sekitar -0.915.
- `AP_RH_ratio` memiliki korelasi sekitar -0.357.

Korelasi tersebut digunakan sebagai analisis awal dan tidak menjadi satu-satunya
dasar pemilihan fitur. Efektivitas fitur tambahan akan diverifikasi melalui
perbandingan performa model pada tahap Machine Learning.

In [10]:
feature_summary = pd.DataFrame({
    "feature": [
        "AT",
        "V",
        "AP",
        "RH",
        "AT_squared",
        "AT_V_interaction",
        "AP_RH_ratio"
    ],
    "correlation_with_PE": [
        -0.947908,
        -0.869900,
        0.518687,
        0.391175,
        -0.914546,
        -0.934471,
        -0.356661
    ]
})

feature_summary.sort_values(
    "correlation_with_PE",
    key=lambda x: abs(x),
    ascending=False
)

,feature,correlation_with_PE
0,AT,-0.947908
5,AT_V_interaction,-0.934471
4,AT_squared,-0.914546
1,V,-0.869900
2,AP,0.518687
3,RH,0.391175
6,AP_RH_ratio,-0.356661


In [11]:
baseline_features = [
    "AT",
    "V",
    "AP",
    "RH"
]

engineered_features = [
    "AT",
    "V",
    "AP",
    "RH",
    "AT_squared",
    "AT_V_interaction",
    "AP_RH_ratio"
]

In [12]:
X_baseline = df[baseline_features].copy()

In [13]:
X_engineered = df.copy()

X_engineered["AT_squared"] = X_engineered["AT"] ** 2
X_engineered["AT_V_interaction"] = (
    X_engineered["AT"] * X_engineered["V"]
)
X_engineered["AP_RH_ratio"] = (
    X_engineered["AP"] / X_engineered["RH"]
)

X_engineered = X_engineered[engineered_features]